# Perceptual Loss and Super-Resolution: Why Pixel MSE Fails

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/perceptual_loss_superres.ipynb)

Build an image super-resolution model using perceptual (feature) loss from a pretrained VGG network. Understand why pixel-level MSE produces blurry outputs and how comparing deep features fixes it.

**Blog post:** [sesen.ai/blog/perceptual-loss-super-resolution](https://sesen.ai/blog/perceptual-loss-super-resolution)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg16_bn, VGG16_BN_Weights
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os, random

device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print(f"Using device: {device}")

## Dataset: Crappify High-Res Images

In [ ]:
def download_pets():
    """Download Oxford-IIIT Pet images."""
    url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
    data_dir = Path("data/pets/images")
    if data_dir.exists() and len(list(data_dir.glob("*.jpg"))) > 100:
        return data_dir
    os.makedirs("data/pets", exist_ok=True)
    import urllib.request, tarfile
    tar_path = "data/pets/images.tar.gz"
    if not Path(tar_path).exists():
        print("Downloading Oxford-IIIT Pet dataset...")
        urllib.request.urlretrieve(url, tar_path)
    with tarfile.open(tar_path) as tar:
        tar.extractall("data/pets")
    return data_dir

class SuperResDataset(Dataset):
    """Pairs of (low-res JPEG, high-res) images."""
    def __init__(self, image_paths, hr_size=128, lr_size=64, quality=60):
        self.paths = image_paths
        self.hr_size = hr_size
        self.lr_size = lr_size
        self.quality = quality
        self.to_tensor = transforms.ToTensor()
        self.normalise = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        # High-res target: center-crop and resize
        w, h = img.size
        s = min(w, h)
        img = img.crop(((w - s) // 2, (h - s) // 2,
                         (w + s) // 2, (h + s) // 2))
        hr = img.resize((self.hr_size, self.hr_size), Image.LANCZOS)
        # Low-res input: downsample + bilinear upsample
        lr = hr.resize((self.lr_size, self.lr_size), Image.BILINEAR)
        lr = lr.resize((self.hr_size, self.hr_size), Image.BILINEAR)
        hr_t = self.normalise(self.to_tensor(hr))
        lr_t = self.normalise(self.to_tensor(lr))
        return lr_t, hr_t

data_dir = download_pets()
all_imgs = sorted([p for p in data_dir.glob("*.jpg")])
print(f"Found {len(all_imgs)} images")
random.seed(42)
random.shuffle(all_imgs)
split = int(0.9 * len(all_imgs))
train_ds = SuperResDataset(all_imgs[:split], hr_size=128, lr_size=64)
valid_ds = SuperResDataset(all_imgs[split:], hr_size=128, lr_size=64)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
valid_dl = DataLoader(valid_ds, batch_size=16, num_workers=2)
print(f"Train: {len(train_ds)}, Valid: {len(valid_ds)}")

## Perceptual (Feature) Loss

In [ ]:
class FeatureLoss(nn.Module):
    """Perceptual loss using VGG-16 features + Gram matrix style loss."""
    def __init__(self):
        super().__init__()
        vgg = vgg16_bn(weights=VGG16_BN_Weights.DEFAULT).features
        vgg.eval()
        for p in vgg.parameters():
            p.requires_grad = False
        # Feature extraction points: just before each MaxPool
        block_ends = [i - 1 for i, m in enumerate(vgg)
                      if isinstance(m, nn.MaxPool2d)]
        # Use blocks 3, 4, 5
        self.ids = block_ends[2:5]
        self.wts = [5.0, 15.0, 2.0]
        # Build sequential sub-networks for each block
        self.blocks = nn.ModuleList()
        prev = 0
        for idx in self.ids:
            self.blocks.append(nn.Sequential(*list(vgg.children())[prev:idx + 1]))
            prev = idx + 1

    def get_features(self, x):
        feats = []
        for block in self.blocks:
            x = block(x)
            feats.append(x)
        return feats

    @staticmethod
    def gram(x):
        n, c, h, w = x.shape
        x = x.view(n, c, -1)
        return (x @ x.transpose(1, 2)) / (c * h * w)

    def forward(self, pred, target):
        loss = F.l1_loss(pred, target)  # pixel loss
        pred_f = self.get_features(pred)
        with torch.no_grad():
            targ_f = self.get_features(target)
        for pf, tf, w in zip(pred_f, targ_f, self.wts):
            loss += F.l1_loss(pf, tf) * w
            loss += F.l1_loss(self.gram(pf), self.gram(tf)) * w ** 2 * 5e3
        return loss

print(f"VGG feature layers: {FeatureLoss().ids}")

## U-Net with ResNet34 Encoder

In [ ]:
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear',
                               align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class SuperResUNet(nn.Module):
    """U-Net with a frozen ResNet34 encoder."""
    def __init__(self):
        super().__init__()
        from torchvision.models import resnet34, ResNet34_Weights
        enc = resnet34(weights=ResNet34_Weights.DEFAULT)
        self.enc0 = nn.Sequential(enc.conv1, enc.bn1, enc.relu)  # 64ch
        self.enc1 = nn.Sequential(enc.maxpool, enc.layer1)       # 64ch
        self.enc2 = enc.layer2   # 128ch
        self.enc3 = enc.layer3   # 256ch
        self.enc4 = enc.layer4   # 512ch

        self.dec4 = UNetBlock(512 + 256, 256)
        self.dec3 = UNetBlock(256 + 128, 128)
        self.dec2 = UNetBlock(128 + 64, 64)
        self.dec1 = UNetBlock(64 + 64, 64)
        self.final = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear',
                         align_corners=False),
            nn.Conv2d(64, 3, 3, padding=1),
        )

    def forward(self, x):
        e0 = self.enc0(x)    # /2
        e1 = self.enc1(e0)   # /4
        e2 = self.enc2(e1)   # /8
        e3 = self.enc3(e2)   # /16
        e4 = self.enc4(e3)   # /32
        d4 = self.dec4(e4, e3)
        d3 = self.dec3(d4, e2)
        d2 = self.dec2(d3, e1)
        d1 = self.dec1(d2, e0)
        return self.final(d1)

model = SuperResUNet().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training with Perceptual Loss

In [ ]:
loss_fn = FeatureLoss().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=1e-3, epochs=10, steps_per_epoch=len(train_dl))

train_losses = []
for epoch in range(10):
    model.train()
    total = 0
    for lr_img, hr_img in train_dl:
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)
        pred = model(lr_img)
        loss = loss_fn(pred, hr_img)
        opt.zero_grad()
        loss.backward()
        opt.step()
        scheduler.step()
        total += loss.item()
    avg_loss = total / len(train_dl)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1:2d} | loss {avg_loss:.4f}")

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 11), train_losses, 'o-', color='#3b82f6', linewidth=2, markersize=8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Perceptual Loss', fontsize=12)
ax.set_title('Training Loss (Feature + Gram + Pixel)', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Train a Pixel MSE Baseline for Comparison

In [ ]:
# Train a second model with plain MSE loss for comparison
model_mse = SuperResUNet().to(device)
opt_mse = torch.optim.Adam(model_mse.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler_mse = torch.optim.lr_scheduler.OneCycleLR(
    opt_mse, max_lr=1e-3, epochs=10, steps_per_epoch=len(train_dl))

for epoch in range(10):
    model_mse.train()
    total = 0
    for lr_img, hr_img in train_dl:
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)
        pred = model_mse(lr_img)
        loss = F.mse_loss(pred, hr_img)
        opt_mse.zero_grad()
        loss.backward()
        opt_mse.step()
        scheduler_mse.step()
        total += loss.item()
    print(f"MSE Epoch {epoch+1:2d} | loss {total/len(train_dl):.4f}")

## Visual Comparison: Low-Res vs MSE vs Perceptual

In [ ]:
# Denormalise for display
inv_norm = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

def to_img(tensor):
    img = inv_norm(tensor).clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    return img

model.eval()
model_mse.eval()

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
cols = ['Low-Res Input', 'MSE Reconstruction', 'Perceptual Reconstruction', 'High-Res Target']
for j, title in enumerate(cols):
    axes[0, j].set_title(title, fontsize=13)

with torch.no_grad():
    for i in range(4):
        lr_img, hr_img = valid_ds[i]
        lr_batch = lr_img.unsqueeze(0).to(device)
        pred_perceptual = model(lr_batch).squeeze(0)
        pred_mse_out = model_mse(lr_batch).squeeze(0)

        axes[i, 0].imshow(to_img(lr_img))
        axes[i, 1].imshow(to_img(pred_mse_out))
        axes[i, 2].imshow(to_img(pred_perceptual))
        axes[i, 3].imshow(to_img(hr_img))
        for j in range(4):
            axes[i, j].axis('off')

plt.suptitle('Super-Resolution: Pixel MSE vs Perceptual Loss', fontsize=16)
plt.tight_layout()
plt.show()

## The Gram Matrix

In [ ]:
# Visualise a Gram matrix from VGG features
sample_lr, sample_hr = valid_ds[0]
feats = loss_fn.get_features(sample_hr.unsqueeze(0).to(device))

gram = FeatureLoss.gram(feats[1])  # block 4 features
gram_np = gram.squeeze(0).cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(gram_np, cmap='RdBu_r')
ax.set_title(f'Gram Matrix (Block 4, {gram_np.shape[0]}x{gram_np.shape[1]} channels)', fontsize=14)
ax.set_xlabel('Channel j')
ax.set_ylabel('Channel i')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## Exercises

1. **Different VGG layers** — Change `block_ends[2:5]` to `block_ends[1:4]` (use earlier layers). How does the output quality change?

2. **Progressive resizing** — After training at 128px, increase to `hr_size=256, lr_size=128` and train for 10 more epochs. Does the extra resolution improve quality?

3. **No Gram loss** — Remove the Gram matrix term from `FeatureLoss.forward`. Compare output texture quality to the full loss.

4. **SSIM metric** — Compute SSIM between predictions and targets for both the MSE and perceptual models. Which has higher SSIM?

5. **Different degradation** — Replace bilinear downsampling with JPEG compression (quality=20). Does the model learn to remove compression artefacts?

## References

- Johnson, J., Alahi, A. & Fei-Fei, L. (2016). [Perceptual Losses for Real-Time Style Transfer and Super-Resolution.](https://arxiv.org/abs/1603.08155)
- Gatys, L.A., Ecker, A.S. & Bethge, M. (2015). [A Neural Algorithm of Artistic Style.](https://arxiv.org/abs/1508.06576)
- fast.ai course: [Practical Deep Learning for Coders, Lesson 7](https://course.fast.ai/).